In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time
from datetime import datetime
import math


class Config:
    WIDTH, HEIGHT = 1280, 720
    APP_NAME = "Virtual Painter Pro - Thesis Edition"
    OUTPUT_FOLDER = "Artworks_Gallery"
    VIDEO_FOLDER = os.path.join(OUTPUT_FOLDER, "Videos")

    # Panels (FIX: più spazio a destra, un po' meno a sinistra)
    LEFT_BAR_W = 175
    RIGHT_BAR_W = 380

    # Buttons / spacing
    BTN_H = 44
    BTN_GAP = 10
    PAD = 12
    DOCK_GAP = 8

    # Titoli sezione (reservati: così non finiscono sotto i tasti)
    TITLE_H = 22
    SECTION_GAP = 12

    # Suavizado del puntero
    SMOOTHING_FACTOR = 0.20

    # Undo history
    MAX_HISTORY = 10

    # Ranges
    BRUSH_MIN, BRUSH_MAX = 3, 50
    ERASER_MIN, ERASER_MAX = 20, 140

    # Video
    VIDEO_FPS = 30.0

    # Fill (básico)
    FILL_MIN_AREA = 30

    # Mandala defaults
    MANDALA_COUNTS = [6, 8, 10, 12, 16]
    MANDALA_MIRROR = True

    # Colors (BGR)
    COLORS = {
        "NERO":    (0, 0, 0),
        "BIANCO":  (255, 255, 255),
        "ROSSO":   (0, 0, 255),
        "VERDE":   (0, 255, 0),
        "BLU":     (255, 0, 0),
        "GIALLO":  (0, 255, 255),
        "VIOLA":   (255, 0, 255),
        "ARANCIO": (0, 69, 255),
        "CIANO":   (255, 255, 0)
    }


def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))

def draw_panel(img, x0, y0, x1, y1, bg=(24, 24, 24), border=(80, 80, 80)):
    cv2.rectangle(img, (x0, y0), (x1, y1), bg, -1)
    cv2.rectangle(img, (x0, y0), (x1, y1), border, 1)

def draw_title(img, text, x, y):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 230, 230), 2)

class UIButton:
    def __init__(self, name, rect, bgr, kind="ACTION", label=None, sublabel=None):
        self.name = name
        self.rect = rect  # (x,y,w,h)
        self.bgr = bgr
        self.kind = kind
        self.label = label if label is not None else name
        self.sublabel = sublabel

    def hit(self, x, y):
        bx, by, bw, bh = self.rect
        return (bx <= x <= bx + bw) and (by <= y <= by + bh)

    def draw(self, img, selected=False, label_override=None, sublabel_override=None):
        bx, by, bw, bh = self.rect

        card = img.copy()
        cv2.rectangle(card, (bx, by), (bx + bw, by + bh), (45, 45, 45), -1)
        cv2.addWeighted(card, 0.55, img, 0.45, 0, img)

        pad = 5
        cv2.rectangle(img, (bx + pad, by + pad), (bx + bw - pad, by + bh - pad), self.bgr, -1)

        border = (255, 255, 255) if selected else (100, 100, 100)
        cv2.rectangle(img, (bx, by), (bx + bw, by + bh), border, 2 if selected else 1)

        txt = label_override if label_override is not None else self.label
        cv2.putText(img, txt, (bx + 10, by + 28),
                    cv2.FONT_HERSHEY_PLAIN, 1.2, (255, 255, 255), 2)

        sub = sublabel_override if sublabel_override is not None else self.sublabel
        if sub:
            cv2.putText(img, sub, (bx + 10, by + bh - 8),
                        cv2.FONT_HERSHEY_PLAIN, 1.0, (235, 235, 235), 1)



class HandDetector:
    def __init__(self, mode=False, max_hands=1, detection_con=0.8, track_con=0.5):
        self.mode = mode
        self.max_hands = max_hands
        self.detection_con = detection_con
        self.track_con = track_con

        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(
            static_image_mode=self.mode,
            max_num_hands=self.max_hands,
            min_detection_confidence=self.detection_con,
            min_tracking_confidence=self.track_con
        )
        self.mpDraw = mp.solutions.drawing_utils
        self.tipIds = [4, 8, 12, 16, 20]

        self.results = None
        self.lmList = []

    def find_hands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        if self.results.multi_hand_landmarks and draw:
            for handLms in self.results.multi_hand_landmarks:
                self.mpDraw.draw_landmarks(img, handLms, self.mpHands.HAND_CONNECTIONS)
        return img

    def find_position(self, img):
        self.lmList = []
        if self.results and self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[0]
            h, w, _ = img.shape
            for idx, lm in enumerate(myHand.landmark):
                cx, cy = int(lm.x * w), int(lm.y * h)
                self.lmList.append([idx, cx, cy])
        return self.lmList

    def fingers_up(self):
        if len(self.lmList) == 0:
            return []
        fingers = []
        if self.lmList[self.tipIds[0]][1] < self.lmList[self.tipIds[0] - 1][1]:
            fingers.append(1)
        else:
            fingers.append(0)
        for i in range(1, 5):
            if self.lmList[self.tipIds[i]][2] < self.lmList[self.tipIds[i] - 2][2]:
                fingers.append(1)
            else:
                fingers.append(0)
        return fingers

    def distance(self, p1, p2):
        if len(self.lmList) == 0:
            return None
        x1, y1 = self.lmList[p1][1], self.lmList[p1][2]
        x2, y2 = self.lmList[p2][1], self.lmList[p2][2]
        return ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5



class PainterEngine:
    def __init__(self):
        self.img_canvas = np.zeros((Config.HEIGHT, Config.WIDTH, 3), np.uint8)
        self.paint_mask = np.zeros((Config.HEIGHT, Config.WIDTH), np.uint8)

        self.undo_stack = []
        self.redo_stack = []

        self.brush_color = Config.COLORS["ROSSO"]
        self.brush_size = 15
        self.eraser_size = 60
        self.is_eraser = False

        self.xp, self.yp = 0, 0
        self.cx, self.cy = 0, 0

        self.active_tool = "ROSSO"
        self.feedback_msg = ""
        self.feedback_timer = 0

        self.save_anim_alpha = 0
        self.last_saved_preview = None

        self.dynamic_color_enabled = False
        self.is_recording = False

        self.fill_enabled = False
        self._fill_used_this_touch = False

        self.mandala_enabled = False
        self.mandala_idx = 1
        self.mandala_mirror = Config.MANDALA_MIRROR

        self.prev_t = None
        self.prev_raw_x = None
        self.prev_raw_y = None
        self.speed_ema = 0.0
        self.speed_alpha = 0.70
        self._last_dt = 1 / 30.0

        self.STOP_DIST_PX = 0.55
        self.STOP_SPEED_PX_S = 1.2
        self.STOP_HOLD_SEC = 0.16
        self._still_time_acc = 0.0

        self.vmin_ema = None
        self.vmax_ema = None
        self.vmin_alpha = 0.05
        self.vmax_alpha = 0.05

        self.intensity_ema = 0.0
        self.intensity_alpha = 0.22
        self.gain = 1.45
        self.min_span = 10.0

        self.intensity_dbg = 0.0
        self.last_dyn_color = self.brush_color

        self.stroke_style = "SOLID"
        self.style_names = ["SOLID", "DOTTED", "DASH", "SPRAY", "MARKER", "NEON", "CHALK"]
        self.dash_phase = 0.0

        self.color_buttons = []
        self.right_buttons = []
        self.style_buttons = []
        self.dock_buttons = []

        # pos titoli (per non farli finire sotto i tasti)
        self.ui_titles = {}

        if not os.path.exists(Config.OUTPUT_FOLDER):
            os.makedirs(Config.OUTPUT_FOLDER)
        if not os.path.exists(Config.VIDEO_FOLDER):
            os.makedirs(Config.VIDEO_FOLDER)

 
    def mandala_count(self):
        return Config.MANDALA_COUNTS[self.mandala_idx % len(Config.MANDALA_COUNTS)]

    def cycle_mandala_count(self, delta):
        self.mandala_idx = (self.mandala_idx + delta) % len(Config.MANDALA_COUNTS)
        self.set_feedback(f"Mandala: {self.mandala_count()}", 25)

    def _rot(self, x, y, ang, cx, cy):
        dx = x - cx
        dy = y - cy
        ca = math.cos(ang)
        sa = math.sin(ang)
        rx = dx * ca - dy * sa
        ry = dx * sa + dy * ca
        return (rx + cx, ry + cy)

    def mandala_segments(self, p1, p2):
        cx0 = Config.LEFT_BAR_W + (Config.WIDTH - Config.LEFT_BAR_W - Config.RIGHT_BAR_W) / 2.0
        cy0 = Config.HEIGHT / 2.0
        n = self.mandala_count()
        out = []

        x1, y1 = p1
        x2, y2 = p2

        for k in range(n):
            ang = 2.0 * math.pi * k / n

            a1 = self._rot(x1, y1, ang, cx0, cy0)
            a2 = self._rot(x2, y2, ang, cx0, cy0)

            p1a = (int(clamp(a1[0], 0, Config.WIDTH - 1)), int(clamp(a1[1], 0, Config.HEIGHT - 1)))
            p2a = (int(clamp(a2[0], 0, Config.WIDTH - 1)), int(clamp(a2[1], 0, Config.HEIGHT - 1)))
            out.append((p1a, p2a))

            if self.mandala_mirror:
                mx1 = (2.0 * cx0 - a1[0], a1[1])
                mx2 = (2.0 * cx0 - a2[0], a2[1])
                p1m = (int(clamp(mx1[0], 0, Config.WIDTH - 1)), int(clamp(mx1[1], 0, Config.HEIGHT - 1)))
                p2m = (int(clamp(mx2[0], 0, Config.WIDTH - 1)), int(clamp(mx2[1], 0, Config.HEIGHT - 1)))
                out.append((p1m, p2m))

        return out

    def draw_segment_with_optional_mandala(self, canvas_img, preview_img, p1, p2, col, thickness, is_eraser):
        if (not self.mandala_enabled) or self.fill_enabled:
            self.draw_stroke(canvas_img, p1, p2, col, thickness)
            self.draw_stroke(preview_img, p1, p2, col, thickness)
            self.update_paint_mask_line(p1, p2, thickness, is_eraser)
            return

        segs = self.mandala_segments(p1, p2)
        for a, b in segs:
            self.draw_stroke(canvas_img, a, b, col, thickness)
            self.draw_stroke(preview_img, a, b, col, thickness)
            self.update_paint_mask_line(a, b, thickness, is_eraser)

   
    def set_feedback(self, msg, duration=30):
        self.feedback_msg = msg
        self.feedback_timer = duration

    def reset_velocity(self):
        self.prev_t = None
        self.prev_raw_x = None
        self.prev_raw_y = None
        self.speed_ema = 0.0
        self._still_time_acc = 0.0
        self.vmin_ema = None
        self.vmax_ema = None
        self.intensity_ema = 0.0
        self.intensity_dbg = 0.0
        self.last_dyn_color = self.brush_color

    def compute_speed(self, x, y):
        now = time.time()
        if self.prev_t is None:
            self.prev_t = now
            self.prev_raw_x, self.prev_raw_y = x, y
            self._last_dt = 1 / 30.0
            return 0.0, 0.0

        dt = now - self.prev_t
        if dt <= 1e-6:
            dt = self._last_dt

        dx = x - self.prev_raw_x
        dy = y - self.prev_raw_y
        dist = (dx * dx + dy * dy) ** 0.5
        speed = dist / dt

        self.speed_ema = self.speed_alpha * speed + (1 - self.speed_alpha) * self.speed_ema

        self.prev_t = now
        self.prev_raw_x, self.prev_raw_y = x, y
        self._last_dt = dt
        return self.speed_ema, dist

    def apply_dynamic_bgr(self, base_bgr, intensity):
        b, g, r = base_bgr
        factor = 0.60 + 0.55 * intensity
        lift = int(35 * intensity)
        b2 = clamp(int(b * factor) + lift, 0, 255)
        g2 = clamp(int(g * factor) + lift, 0, 255)
        r2 = clamp(int(r * factor) + lift, 0, 255)
        return (b2, g2, r2)

    def get_dynamic_color(self, raw_x, raw_y):
        sp, dist = self.compute_speed(raw_x, raw_y)

        if dist < self.STOP_DIST_PX and sp < self.STOP_SPEED_PX_S:
            self._still_time_acc += self._last_dt
        else:
            self._still_time_acc = 0.0

        if self._still_time_acc >= self.STOP_HOLD_SEC:
            self.intensity_dbg = 0.0
            return self.last_dyn_color

        if self.vmin_ema is None:
            self.vmin_ema = sp
            self.vmax_ema = sp

        self.vmin_ema = (1 - self.vmin_alpha) * self.vmin_ema + self.vmin_alpha * min(self.vmin_ema, sp)
        self.vmax_ema = (1 - self.vmax_alpha) * self.vmax_ema + self.vmax_alpha * max(self.vmax_ema, sp)

        span = max(self.min_span, self.vmax_ema - self.vmin_ema)
        t = (sp - self.vmin_ema) / span
        t = clamp(t, 0.0, 1.0)
        t = clamp(t * self.gain, 0.0, 1.0)
        target_intensity = t ** 0.95

        self.intensity_ema = (1 - self.intensity_alpha) * self.intensity_ema + self.intensity_alpha * target_intensity
        self.intensity_dbg = float(self.intensity_ema)

        col = self.apply_dynamic_bgr(self.brush_color, self.intensity_ema)
        self.last_dyn_color = col
        return col

  
    def save_state_for_undo(self):
        if len(self.undo_stack) >= Config.MAX_HISTORY:
            self.undo_stack.pop(0)
        self.undo_stack.append((self.img_canvas.copy(), self.paint_mask.copy()))
        self.redo_stack.clear()

    def perform_undo(self):
        if self.undo_stack:
            self.redo_stack.append((self.img_canvas.copy(), self.paint_mask.copy()))
            self.img_canvas, self.paint_mask = self.undo_stack.pop()
            self.set_feedback("Undo", 25)

    def perform_redo(self):
        if self.redo_stack:
            self.undo_stack.append((self.img_canvas.copy(), self.paint_mask.copy()))
            self.img_canvas, self.paint_mask = self.redo_stack.pop()
            self.set_feedback("Redo", 25)

    def change_brush(self, delta):
        self.brush_size = clamp(self.brush_size + delta, Config.BRUSH_MIN, Config.BRUSH_MAX)
        self.set_feedback(f"Brush: {self.brush_size}", 20)

    def change_eraser(self, delta):
        self.eraser_size = clamp(self.eraser_size + delta, Config.ERASER_MIN, Config.ERASER_MAX)
        self.set_feedback(f"Eraser: {self.eraser_size}", 20)

    def save_artwork(self):
        filename = f"{Config.OUTPUT_FOLDER}/Art_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        cv2.imwrite(filename, self.img_canvas)
        self.save_anim_alpha = 255
        self.last_saved_preview = cv2.resize(self.img_canvas, (160, 90))
        self.set_feedback("Salvato!", 60)

   
    def flood_fill_basic(self, x, y, fill_col):
        if x < 0 or y < 0 or x >= Config.WIDTH or y >= Config.HEIGHT:
            return False
        if self.paint_mask[y, x] > 0:
            return False

        tmp = np.zeros((Config.HEIGHT, Config.WIDTH), np.uint8)
        tmp[self.paint_mask > 0] = 255

        mask = np.zeros((Config.HEIGHT + 2, Config.WIDTH + 2), np.uint8)
        flags = 4 | (128 << 8)
        cv2.floodFill(tmp, mask, (int(x), int(y)), 128, flags=flags)

        region = (tmp == 128)
        area = int(region.sum())
        if area < Config.FILL_MIN_AREA:
            return False

        self.img_canvas[region] = fill_col
        self.paint_mask[region] = 255
        return True

 
    def _draw_solid(self, img, p1, p2, col, thickness):
        cv2.line(img, p1, p2, col, thickness, lineType=cv2.LINE_AA)

    def _draw_dotted(self, img, p1, p2, col, thickness):
        x1, y1 = p1
        x2, y2 = p2
        dx = x2 - x1
        dy = y2 - y1
        dist = (dx * dx + dy * dy) ** 0.5
        if dist < 1:
            cv2.circle(img, p2, max(1, thickness // 2), col, -1, lineType=cv2.LINE_AA)
            return
        step = max(3, thickness * 2)
        n = int(dist / step) + 1
        for i in range(n + 1):
            t = i / max(1, n)
            x = int(x1 + dx * t)
            y = int(y1 + dy * t)
            cv2.circle(img, (x, y), max(1, thickness // 2), col, -1, lineType=cv2.LINE_AA)

    def _draw_dash(self, img, p1, p2, col, thickness, phase):
        x1, y1 = p1
        x2, y2 = p2
        dx = x2 - x1
        dy = y2 - y1
        dist = (dx * dx + dy * dy) ** 0.5
        if dist < 1:
            return phase

        dash_len = max(40, thickness * 8)
        gap_len  = max(35, thickness * 7)
        period = dash_len + gap_len

        ux = dx / dist
        uy = dy / dist

        s = phase % period
        pos = -s

        while pos < dist:
            dash_start = max(pos, 0.0)
            dash_end   = min(pos + dash_len, dist)

            if dash_end > 0 and dash_start < dist:
                xa = int(x1 + ux * dash_start)
                ya = int(y1 + uy * dash_start)
                xb = int(x1 + ux * dash_end)
                yb = int(y1 + uy * dash_end)

                cv2.line(img, (xa, ya), (xb, yb), col, thickness, lineType=cv2.LINE_AA)
                rad = max(1, thickness // 2)
                cv2.circle(img, (xa, ya), rad, col, -1, lineType=cv2.LINE_AA)
                cv2.circle(img, (xb, yb), rad, col, -1, lineType=cv2.LINE_AA)

            pos += period

        return phase + dist

    def _draw_spray(self, img, p1, p2, col, thickness):
        x1, y1 = p1
        x2, y2 = p2
        dx = x2 - x1
        dy = y2 - y1
        dist = (dx * dx + dy * dy) ** 0.5
        if dist < 1:
            dist = 1
        step = 6
        n = int(dist / step) + 1
        radius = int(max(6, thickness * 1.2))
        pts_per_step = int(16 + thickness * 1.2)
        for i in range(n + 1):
            tt = i / max(1, n)
            cx = int(x1 + dx * tt)
            cy = int(y1 + dy * tt)
            for _ in range(pts_per_step):
                ox = int(np.random.normal(0, radius * 0.35))
                oy = int(np.random.normal(0, radius * 0.35))
                px = clamp(cx + ox, 0, Config.WIDTH - 1)
                py = clamp(cy + oy, 0, Config.HEIGHT - 1)
                img[py, px] = col
                self.paint_mask[py, px] = 255

    def _draw_marker(self, img, p1, p2, col, thickness):
        overlay = img.copy()
        cv2.line(overlay, p1, p2, col, thickness, lineType=cv2.LINE_AA)
        cv2.addWeighted(overlay, 0.35, img, 0.65, 0, img)

    def _draw_neon(self, img, p1, p2, col, thickness):
        b, g, r = col
        glow = (clamp(int(b * 0.55), 0, 255),
                clamp(int(g * 0.55), 0, 255),
                clamp(int(r * 0.55), 0, 255))
        overlay = img.copy()
        cv2.line(overlay, p1, p2, glow, thickness + 10, lineType=cv2.LINE_AA)
        cv2.GaussianBlur(overlay, (0, 0), 6, overlay)
        cv2.addWeighted(overlay, 0.45, img, 0.55, 0, img)
        cv2.line(img, p1, p2, col, thickness, lineType=cv2.LINE_AA)

    def _draw_chalk(self, img, p1, p2, col, thickness):
        overlay = img.copy()
        cv2.line(overlay, p1, p2, col, thickness, lineType=cv2.LINE_AA)
        cv2.addWeighted(overlay, 0.28, img, 0.72, 0, img)

        x1, y1 = p1
        x2, y2 = p2
        dx = x2 - x1
        dy = y2 - y1
        dist = (dx * dx + dy * dy) ** 0.5
        if dist < 1:
            dist = 1
        step = 10
        n = int(dist / step) + 1
        radius = int(max(4, thickness * 0.8))
        pts = int(6 + thickness)

        for i in range(n + 1):
            tt = i / max(1, n)
            cx = int(x1 + dx * tt)
            cy = int(y1 + dy * tt)
            for _ in range(pts):
                ox = int(np.random.normal(0, radius * 0.45))
                oy = int(np.random.normal(0, radius * 0.45))
                px = clamp(cx + ox, 0, Config.WIDTH - 1)
                py = clamp(cy + oy, 0, Config.HEIGHT - 1)
                img[py, px] = (clamp(col[0] + np.random.randint(-25, 26), 0, 255),
                               clamp(col[1] + np.random.randint(-25, 26), 0, 255),
                               clamp(col[2] + np.random.randint(-25, 26), 0, 255))
                self.paint_mask[py, px] = 255

    def draw_stroke(self, img, p1, p2, col, thickness):
        s = self.stroke_style
        if s == "SOLID":
            self._draw_solid(img, p1, p2, col, thickness)
        elif s == "DOTTED":
            self._draw_dotted(img, p1, p2, col, thickness)
        elif s == "DASH":
            self.dash_phase = self._draw_dash(img, p1, p2, col, thickness, self.dash_phase)
        elif s == "SPRAY":
            self._draw_spray(img, p1, p2, col, thickness)
        elif s == "MARKER":
            self._draw_marker(img, p1, p2, col, thickness)
        elif s == "NEON":
            self._draw_neon(img, p1, p2, col, thickness)
        elif s == "CHALK":
            self._draw_chalk(img, p1, p2, col, thickness)
        else:
            self._draw_solid(img, p1, p2, col, thickness)

    def update_paint_mask_line(self, p1, p2, thickness, is_eraser):
        val = 0 if is_eraser else 255
        cv2.line(self.paint_mask, p1, p2, val, thickness, lineType=cv2.LINE_AA)


    def build_ui(self):
        self.color_buttons = []
        self.right_buttons = []
        self.style_buttons = []
        self.dock_buttons = []
        self.ui_titles = {}

        # LEFT panel (palette + dock). Dock resta a sinistra come prima.
        lx0 = 0
        y = Config.PAD + Config.TITLE_H + 12  # spazio titolo PALETTE
        palette = [
            ("ROSSO",  Config.COLORS["ROSSO"]),
            ("VERDE",  Config.COLORS["VERDE"]),
            ("BLU",    Config.COLORS["BLU"]),
            ("GIALLO", Config.COLORS["GIALLO"]),
            ("VIOLA",  Config.COLORS["VIOLA"]),
            ("CIANO",  Config.COLORS["CIANO"]),
        ]
        for name, col in palette:
            rect = (lx0 + Config.PAD, y, Config.LEFT_BAR_W - 2 * Config.PAD, Config.BTN_H)
            self.color_buttons.append(UIButton(name, rect, col, "COLOR", name))
            y += Config.BTN_H + max(6, Config.BTN_GAP - 2)

        # DOCK bottom-left
        dock_w = Config.LEFT_BAR_W - 2 * Config.PAD
        dock_x = lx0 + Config.PAD
        col_w = (dock_w - Config.DOCK_GAP) // 2
        row_h = 40
        rows = 3
        dock_bottom = Config.HEIGHT - Config.PAD
        dock_top = dock_bottom - rows * row_h - (rows - 1) * Config.DOCK_GAP

        def rrect(r, c):
            x = dock_x + c * (col_w + Config.DOCK_GAP)
            yy = dock_top + r * (row_h + Config.DOCK_GAP)
            return (x, yy, col_w, row_h)

        self.btn_eraser = UIButton("GOMMA", rrect(0, 0), (55, 55, 55), "TOOL", "GOMMA")
        self.btn_clear = UIButton("CLEAR", rrect(0, 1), (70, 70, 160), "ACTION", "CLEAR")
        self.btn_undo = UIButton("UNDO", rrect(1, 0), (110, 110, 110), "ACTION", "UNDO")
        self.btn_redo = UIButton("REDO", rrect(1, 1), (120, 120, 120), "ACTION", "REDO")
        self.btn_size_plus = UIButton("SIZE+", rrect(2, 0), (40, 160, 200), "ACTION", "SIZE +")
        self.btn_size_minus = UIButton("SIZE-", rrect(2, 1), (40, 160, 200), "ACTION", "SIZE -")

        self.dock_buttons = [
            self.btn_eraser, self.btn_clear,
            self.btn_undo, self.btn_redo,
            self.btn_size_plus, self.btn_size_minus
        ]

        # RIGHT panel
        rx0 = Config.WIDTH - Config.RIGHT_BAR_W
        x = rx0 + Config.PAD
        w = Config.RIGHT_BAR_W - 2 * Config.PAD
        half = (w - 10) // 2

        # blocchi con titolo "riservato"
        def section(title, y_start):
            self.ui_titles[title] = y_start + 18  # baseline testo titolo
            return y_start + Config.TITLE_H + 10

        yR = Config.PAD

        yR = section("EXPORT", yR)
        self.btn_save = UIButton("SAVE", (x, yR, w, Config.BTN_H), (50, 200, 50), "ACTION", "SALVA PNG", "immagine finale")
        yR += Config.BTN_H + Config.SECTION_GAP

        yR = section("MODES", yR)
        self.btn_dyn_on  = UIButton("DYN_ON",  (x, yR, half, Config.BTN_H), (80, 80, 80), "ACTION", "DYN ON")
        self.btn_dyn_off = UIButton("DYN_OFF", (x + half + 10, yR, half, Config.BTN_H), (60, 60, 60), "ACTION", "DYN OFF")
        yR += Config.BTN_H + 8

        self.btn_rec_on  = UIButton("REC_ON",  (x, yR, half, Config.BTN_H), (55, 55, 200), "ACTION", "REC ON")
        self.btn_rec_off = UIButton("REC_OFF", (x + half + 10, yR, half, Config.BTN_H), (45, 45, 140), "ACTION", "REC OFF")
        yR += Config.BTN_H + 8

        self.btn_fill_on  = UIButton("FILL_ON",  (x, yR, half, Config.BTN_H), (90, 70, 10), "ACTION", "FILL ON")
        self.btn_fill_off = UIButton("FILL_OFF", (x + half + 10, yR, half, Config.BTN_H), (60, 45, 10), "ACTION", "FILL OFF")
        yR += Config.BTN_H + Config.SECTION_GAP

        yR = section("MANDALA", yR)
        self.btn_mand_on  = UIButton("MAND_ON",  (x, yR, half, Config.BTN_H), (70, 120, 170), "ACTION", "MAND ON")
        self.btn_mand_off = UIButton("MAND_OFF", (x + half + 10, yR, half, Config.BTN_H), (50, 80, 110), "ACTION", "MAND OFF")
        yR += Config.BTN_H + 8

        self.btn_mand_minus = UIButton("MAND_-", (x, yR, half, Config.BTN_H), (60, 100, 140), "ACTION", "SECT -")
        self.btn_mand_plus  = UIButton("MAND_+", (x + half + 10, yR, half, Config.BTN_H), (60, 100, 140), "ACTION", "SECT +")
        yR += Config.BTN_H + Config.SECTION_GAP

        # QUIT bottom (prima calcolo spazio per STYLE)
        quit_h = Config.BTN_H
        quit_y = Config.HEIGHT - Config.PAD - quit_h
        self.btn_quit = UIButton("QUIT", (x, quit_y, w, quit_h), (30, 30, 150), "ACTION", "QUIT")

        # STYLE (grid completo, ma adattivo: non si accavalla)
        yR = section("STYLE", yR)

        # area disponibile tra yR e quit
        available = max(60, quit_y - 12 - yR)

        # layout 2 colonne, 4 righe max (per 7 stili)
        cols = 2
        rows = int(math.ceil(len(self.style_names) / cols))

        # scegli altezza e gap per farlo entrare SEMPRE
        gap = 8
        btn_h = Config.BTN_H
        need = rows * btn_h + (rows - 1) * gap

        if need > available:
            # stringi SOLO gli style buttons (senza togliere nulla)
            gap = 6
            btn_h = int((available - (rows - 1) * gap) / rows)
            btn_h = clamp(btn_h, 30, Config.BTN_H)

            need = rows * btn_h + (rows - 1) * gap
            if need > available:
                # ultima ancora: riduci gap al minimo
                gap = 4
                btn_h = int((available - (rows - 1) * gap) / rows)
                btn_h = clamp(btn_h, 28, Config.BTN_H)

        col_w2 = (w - 10) // 2
        for i, style_name in enumerate(self.style_names):
            r = i // 2
            c = i % 2
            bx = x + c * (col_w2 + 10)
            by = yR + r * (btn_h + gap)
            rect = (bx, by, col_w2, btn_h)
            self.style_buttons.append(UIButton(style_name, rect, (105, 85, 85), "STYLE", style_name))

        # right_buttons (tutti, nessuno tolto)
        self.right_buttons = [
            self.btn_save,
            self.btn_dyn_on, self.btn_dyn_off,
            self.btn_rec_on, self.btn_rec_off,
            self.btn_fill_on, self.btn_fill_off,
            self.btn_mand_on, self.btn_mand_off,
            self.btn_mand_minus, self.btn_mand_plus,
            self.btn_quit
        ]

    def draw_ui(self, img):
        self.build_ui()

        draw_panel(img, 0, 0, Config.LEFT_BAR_W, Config.HEIGHT)
        draw_panel(img, Config.WIDTH - Config.RIGHT_BAR_W, 0, Config.WIDTH, Config.HEIGHT)

        # Titoli (sempre sopra i blocchi, mai sotto i tasti)
        draw_title(img, "PALETTE", Config.PAD, Config.PAD + 18)
        draw_title(img, "DOCK", Config.PAD, self.dock_buttons[0].rect[1] - 10)

        rx0 = Config.WIDTH - Config.RIGHT_BAR_W
        for k, y in self.ui_titles.items():
            draw_title(img, k, rx0 + Config.PAD, y)

        for b in self.color_buttons:
            b.draw(img, selected=(b.name == self.active_tool))

        for db in self.dock_buttons:
            if db.name == "GOMMA":
                db.draw(img, selected=self.is_eraser)
            else:
                db.draw(img, selected=(db.name == self.active_tool))

        self.btn_save.draw(img, selected=False)

        self.btn_dyn_on.draw(img, selected=self.dynamic_color_enabled)
        self.btn_dyn_off.draw(img, selected=(not self.dynamic_color_enabled))

        self.btn_rec_on.draw(img, selected=self.is_recording)
        self.btn_rec_off.draw(img, selected=(not self.is_recording))

        self.btn_fill_on.draw(img, selected=self.fill_enabled)
        self.btn_fill_off.draw(img, selected=(not self.fill_enabled))

        self.btn_mand_on.draw(img, selected=self.mandala_enabled)
        self.btn_mand_off.draw(img, selected=(not self.mandala_enabled))
        self.btn_mand_minus.draw(img, selected=False, sublabel_override=f"N={self.mandala_count()}")
        self.btn_mand_plus.draw(img, selected=False, sublabel_override=f"N={self.mandala_count()}")

        if self.is_recording:
            bx, by, bw, bh = self.btn_rec_on.rect
            cv2.circle(img, (bx + bw - 16, by + 16), 7, (0, 0, 255), -1)

        for sb in self.style_buttons:
            sb.draw(img, selected=(sb.name == self.stroke_style))

        self.btn_quit.draw(img, selected=(self.active_tool == "QUIT"))

        cx = Config.LEFT_BAR_W + 12
        dyn = "ON" if self.dynamic_color_enabled else "OFF"
        rec = "ON" if self.is_recording else "OFF"
        fill = "ON" if self.fill_enabled else "OFF"
        mand = "ON" if self.mandala_enabled else "OFF"
        cv2.putText(img,
                    f"Style: {self.stroke_style}   DYN:{dyn}   REC:{rec}   FILL:{fill}   MAND:{mand} N={self.mandala_count()}",
                    (cx, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (220, 220, 220), 2)
        cv2.putText(img,
                    f"Speed: {int(self.speed_ema)} px/s   Intensity: {self.intensity_dbg:.2f}",
                    (cx, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1)

    def update_feedback(self, img):
        if self.save_anim_alpha > 0:
            overlay = img.copy()
            cv2.rectangle(overlay, (0, 0), (Config.WIDTH, Config.HEIGHT), (255, 255, 255), -1)
            cv2.addWeighted(overlay, self.save_anim_alpha / 255.0, img, 1 - self.save_anim_alpha / 255.0, 0, img)
            self.save_anim_alpha -= 20

        if self.feedback_timer > 0:
            cv2.putText(img, self.feedback_msg,
                        (Config.WIDTH // 2 - 170, Config.HEIGHT // 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
            self.feedback_timer -= 1

        if self.last_saved_preview is not None and self.feedback_timer > 0:
            h, w, _ = self.last_saved_preview.shape
            y_off = Config.HEIGHT - h - 10
            x_off = Config.WIDTH - w - 10
            cv2.rectangle(img, (x_off - 2, y_off - 2), (Config.WIDTH - 8, Config.HEIGHT - 8), (255, 255, 255), 2)
            img[y_off:y_off + h, x_off:x_off + w] = self.last_saved_preview



def main():
    cap = cv2.VideoCapture(0)
    cap.set(3, Config.WIDTH)
    cap.set(4, Config.HEIGHT)

    detector = HandDetector(max_hands=1)
    engine = PainterEngine()

    pTime = 0
    drawing_mode_active = False

    # Pinch (size)
    pinch_active = False
    pinch_ref = None
    pinch_dist_smooth = None
    PINCH_START = 45
    PINCH_END = 65
    PINCH_ALPHA = 0.25
    PINCH_STEP = 6

    # Video writer
    video_writer = None
    recording = False

    def start_recording():
        nonlocal video_writer, recording
        if recording:
            return
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        fname = os.path.join(Config.VIDEO_FOLDER, f"Canvas_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4")
        video_writer = cv2.VideoWriter(fname, fourcc, Config.VIDEO_FPS, (Config.WIDTH, Config.HEIGHT))
        recording = True
        engine.is_recording = True
        engine.set_feedback("REC start", 35)

    def stop_recording():
        nonlocal video_writer, recording
        if not recording:
            return
        recording = False
        engine.is_recording = False
        if video_writer is not None:
            video_writer.release()
            video_writer = None
        engine.set_feedback("REC stop", 35)

    running = True
    while running:
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        _ = cv2.waitKey(1) & 0xFF

        frame = detector.find_hands(frame)
        lmList = detector.find_position(frame)

        engine.draw_ui(frame)

        if len(lmList) != 0:
            x1, y1 = lmList[8][1:]
            x2, y2 = lmList[12][1:]
            fingers = detector.fingers_up()

            # PINCH size
            if fingers and fingers[0] and fingers[1] and (not fingers[2]) and (not fingers[3]) and (not fingers[4]):
                d = detector.distance(4, 8)
                if d is not None:
                    if pinch_dist_smooth is None:
                        pinch_dist_smooth = d
                    pinch_dist_smooth = PINCH_ALPHA * d + (1 - PINCH_ALPHA) * pinch_dist_smooth

                    if (not pinch_active) and pinch_dist_smooth < PINCH_START:
                        pinch_active = True
                        pinch_ref = pinch_dist_smooth

                    if pinch_active:
                        delta = pinch_dist_smooth - pinch_ref
                        if abs(delta) > 2:
                            step = int(delta / PINCH_STEP)
                            if step != 0:
                                if engine.is_eraser:
                                    engine.change_eraser(step * 4)
                                else:
                                    engine.change_brush(step)
                                pinch_ref = pinch_dist_smooth
                        cv2.circle(frame, (x1, y1), 10, (255, 255, 255), -1)

                    if pinch_active and pinch_dist_smooth > PINCH_END:
                        pinch_active = False
                        pinch_ref = None
            else:
                pinch_active = False
                pinch_ref = None
                pinch_dist_smooth = None

            # SELECTION (2 dita)
            if fingers and len(fingers) >= 3 and fingers[1] and fingers[2]:
                engine.xp, engine.yp = 0, 0
                drawing_mode_active = False
                engine.reset_velocity()
                engine._fill_used_this_touch = False

                cv2.rectangle(frame, (x1, y1 - 25), (x2, y2 + 25), engine.brush_color, -1)

                for b in engine.color_buttons:
                    if b.hit(x1, y1):
                        engine.active_tool = b.name
                        engine.brush_color = b.bgr
                        engine.is_eraser = False
                        engine.last_dyn_color = engine.brush_color
                        engine.reset_velocity()

                for db in engine.dock_buttons:
                    if db.hit(x1, y1):
                        engine.active_tool = db.name

                        if db.name == "GOMMA":
                            engine.is_eraser = True
                            engine.reset_velocity()

                        elif db.name == "CLEAR":
                            engine.save_state_for_undo()
                            engine.img_canvas[:] = 0
                            engine.paint_mask[:] = 0
                            engine.set_feedback("Pulito", 35)

                        elif db.name == "UNDO":
                            if engine.feedback_timer == 0:
                                engine.perform_undo()

                        elif db.name == "REDO":
                            if engine.feedback_timer == 0:
                                engine.perform_redo()

                        elif db.name == "SIZE+":
                            if engine.is_eraser:
                                engine.change_eraser(+10)
                            else:
                                engine.change_brush(+2)

                        elif db.name == "SIZE-":
                            if engine.is_eraser:
                                engine.change_eraser(-10)
                            else:
                                engine.change_brush(-2)

                for rb in engine.right_buttons:
                    if rb.hit(x1, y1):
                        engine.active_tool = rb.name

                        if rb.name == "SAVE":
                            if engine.feedback_timer == 0:
                                engine.save_artwork()

                        elif rb.name == "DYN_ON":
                            engine.dynamic_color_enabled = True
                            engine.last_dyn_color = engine.brush_color
                            engine.reset_velocity()
                            engine.set_feedback("DYN: ON", 30)

                        elif rb.name == "DYN_OFF":
                            engine.dynamic_color_enabled = False
                            engine.reset_velocity()
                            engine.intensity_dbg = 0.0
                            engine.set_feedback("DYN: OFF", 30)

                        elif rb.name == "REC_ON":
                            if not recording:
                                start_recording()

                        elif rb.name == "REC_OFF":
                            if recording:
                                stop_recording()

                        elif rb.name == "FILL_ON":
                            engine.fill_enabled = True
                            engine.set_feedback("FILL: ON", 30)

                        elif rb.name == "FILL_OFF":
                            engine.fill_enabled = False
                            engine.set_feedback("FILL: OFF", 30)

                        elif rb.name == "MAND_ON":
                            engine.mandala_enabled = True
                            engine.set_feedback(f"MAND: ON (N={engine.mandala_count()})", 30)

                        elif rb.name == "MAND_OFF":
                            engine.mandala_enabled = False
                            engine.set_feedback("MAND: OFF", 30)

                        elif rb.name == "MAND_+":
                            engine.cycle_mandala_count(+1)

                        elif rb.name == "MAND_-":
                            engine.cycle_mandala_count(-1)

                        elif rb.name == "QUIT":
                            running = False

                for sb in engine.style_buttons:
                    if sb.hit(x1, y1):
                        engine.stroke_style = sb.name
                        engine.active_tool = "STYLE"
                        engine.set_feedback(f"Style: {engine.stroke_style}", 25)

            # DRAW (indice)
            elif fingers and len(fingers) >= 3 and fingers[1] and not fingers[2]:
                in_left = x1 < Config.LEFT_BAR_W
                in_right = x1 > (Config.WIDTH - Config.RIGHT_BAR_W)
                safe_draw = (not in_left) and (not in_right)

                if safe_draw:
                    if not drawing_mode_active:
                        engine.save_state_for_undo()
                        drawing_mode_active = True
                        engine.xp, engine.yp = x1, y1
                        engine.cx, engine.cy = x1, y1
                        engine.reset_velocity()
                        engine.dash_phase = 0.0
                        engine._fill_used_this_touch = False

                    engine.cx = int(engine.cx * (1 - Config.SMOOTHING_FACTOR) + x1 * Config.SMOOTHING_FACTOR)
                    engine.cy = int(engine.cy * (1 - Config.SMOOTHING_FACTOR) + y1 * Config.SMOOTHING_FACTOR)

                    thickness = engine.eraser_size if engine.is_eraser else int(engine.brush_size)

                    if engine.is_eraser:
                        col = (0, 0, 0)
                        engine.compute_speed(x1, y1)
                    else:
                        if engine.dynamic_color_enabled:
                            col = engine.get_dynamic_color(x1, y1)
                        else:
                            engine.compute_speed(x1, y1)
                            engine.intensity_dbg = 0.0
                            col = engine.brush_color

                    cv2.circle(frame, (engine.cx, engine.cy), 12, col, -1)

                    if engine.fill_enabled:
                        if not engine._fill_used_this_touch:
                            engine.save_state_for_undo()
                            ok = engine.flood_fill_basic(engine.cx, engine.cy, col)
                            engine._fill_used_this_touch = True
                            engine.set_feedback("Fill" if ok else "No area", 22 if ok else 18)
                        engine.xp, engine.yp = engine.cx, engine.cy
                    else:
                        if engine.xp != 0:
                            engine.draw_segment_with_optional_mandala(
                                engine.img_canvas, frame,
                                (engine.xp, engine.yp), (engine.cx, engine.cy),
                                col, thickness, engine.is_eraser
                            )
                        engine.xp, engine.yp = engine.cx, engine.cy

                else:
                    engine.xp, engine.yp = 0, 0
                    drawing_mode_active = False
                    engine.reset_velocity()
                    engine._fill_used_this_touch = False
            else:
                engine.xp, engine.yp = 0, 0
                drawing_mode_active = False
                engine.reset_velocity()
                engine._fill_used_this_touch = False

        mask = engine.paint_mask > 0
        frame[mask] = engine.img_canvas[mask]

        engine.draw_ui(frame)
        engine.update_feedback(frame)

        cTime = time.time()
        fps = 1 / (cTime - pTime) if (cTime - pTime) > 0 else 30
        pTime = cTime
        cv2.putText(frame, f'FPS: {int(fps)}',
                    (Config.WIDTH // 2 - 30, Config.HEIGHT - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 100, 100), 1)

        if recording and video_writer is not None:
            video_writer.write(engine.img_canvas)

        cv2.imshow(Config.APP_NAME, frame)

    if recording:
        stop_recording()
    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

